# Mask Quality Analysis

Notebook 05 computes general patch quality metrics covering both the echogram signal and the bottom-detection mask. This notebook focuses exclusively on the mask — the goal is to find labels that are wrong or unreliable before they corrupt training.

Each mask is a binary array marking everything at and below the detected seabed. The bottom line is extracted as the first nonzero row per column. A good mask has a smooth, continuous bottom line that sits on a visible acoustic boundary and is consistent across both frequencies. A bad mask may have gaps, jagged jumps, no physical boundary underneath, or disagree between 38 kHz and 200 kHz.

In [ ]:
import sys
sys.path.insert(0, '../../..')

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import ndimage, stats
from tqdm.notebook import tqdm

PATCH_DIR = Path('/Users/axelhagen/Master/Datasets/patch_cache')
OUTPUT_CSV = Path('/Users/axelhagen/Master/Datasets/mask_quality_metrics.csv')
FLAGGED_JSON = Path('/Users/axelhagen/Master/Datasets/flagged_bad_masks.json')

# Existing patch-level metrics from notebook 05 (for correlation analysis)
NB05_CSV = Path('/Users/axelhagen/Master/Datasets/patch_quality_metrics.csv')

npz_files = sorted(PATCH_DIR.glob('*.npz'))
print(f'Found {len(npz_files)} npz files in {PATCH_DIR}')

## Mask metric functions

All metrics target specific mask failure modes:

- **bottom_continuity** — fraction of consecutive ping pairs where both have a valid bottom. Low values mean large detection gaps that were filled by interpolation.
- **bottom_curvature** — mean absolute second derivative of the bottom line. High values indicate a jagged, noisy detection that doesn't follow a smooth seabed.
- **mask_edge_intensity** — mean echogram intensity (ch0) at the exact bottom row. A real seabed return should be bright; a low value suggests the mask boundary doesn't sit on a physical feature.
- **cross_freq_bottom_diff** — mean absolute difference in bottom position when estimated independently from ch0 (38 kHz) and ch1 (200 kHz). Large disagreement suggests one frequency's detection went wrong.
- **below_bottom_decay** — mean intensity slope in a band below the bottom (should decrease away from the boundary). Non-decaying intensity suggests a false bottom above the real one.
- **above_bottom_cv** — coefficient of variation of water-column intensity above the bottom. High CV indicates scattering layers or noise that can confuse detectors.
- **mask_connectivity** — number of connected components in the binary mask. A clean "below" mask should have exactly 1 component; more means holes or fragmentation.
- **bottom_snr** — signal-to-noise ratio at the boundary: mean intensity in a 5-pixel band below the bottom minus mean intensity in a 5-pixel band above.

In [ ]:
def get_bottom_line(mask):
    """Extract per-ping bottom depth index from a binary mask.

    For each column, finds the first row where mask == 1 (the top of the
    'below-bottom' region). Returns -1 for columns with no bottom detected.
    """
    H, W = mask.shape
    bottom = np.full(W, -1, dtype=int)
    for j in range(W):
        col = mask[:, j]
        if col.any():
            bottom[j] = np.argmax(col)
    return bottom


def estimate_bottom_from_channel(channel, band=20):
    """Estimate bottom position per ping from a single-channel echogram.

    For each column, finds the row with maximum intensity in the lower 80%
    of the image (skip the near-surface region). This gives a rough
    frequency-independent bottom estimate for cross-frequency comparison.
    """
    H, W = channel.shape
    start_row = H // 5  # skip top 20%
    bottom = np.full(W, -1, dtype=int)
    for j in range(W):
        col = channel[start_row:, j]
        if col.max() > 0.1:  # need some signal
            bottom[j] = start_row + np.argmax(col)
    return bottom


def compute_mask_metrics(patch, mask):
    """Compute mask-focused quality metrics for a single patch + mask pair.

    Parameters
    ----------
    patch : ndarray, shape (3, H, W), float32
    mask  : ndarray, shape (H, W), uint8

    Returns
    -------
    metrics : dict
    """
    ch0 = patch[0]  # 38 kHz, normalised [0, 1]
    ch1 = patch[1]  # 200 kHz, normalised [0, 1]
    H, W = ch0.shape

    bottom = get_bottom_line(mask)
    valid = bottom >= 0
    n_valid = valid.sum()

    m = {}

    # --- Bottom line continuity ---
    # Fraction of consecutive pings where both have a detected bottom.
    if W > 1:
        both_valid = valid[:-1] & valid[1:]
        m['bottom_continuity'] = float(both_valid.sum() / (W - 1))
    else:
        m['bottom_continuity'] = np.nan

    # --- Bottom curvature (second derivative) ---
    if n_valid > 2:
        valid_depths = bottom[valid].astype(float)
        d2 = np.diff(valid_depths, n=2)
        m['bottom_curvature'] = float(np.mean(np.abs(d2)))
    else:
        m['bottom_curvature'] = np.nan

    # --- Edge intensity: mean ch0 value at the exact bottom row ---
    if n_valid > 0:
        intensities = []
        for j in np.where(valid)[0]:
            b = bottom[j]
            if 0 <= b < H:
                intensities.append(ch0[b, j])
        m['mask_edge_intensity'] = float(np.mean(intensities)) if intensities else np.nan
    else:
        m['mask_edge_intensity'] = np.nan

    # --- Cross-frequency bottom difference ---
    # Compare mask bottom to an independent estimate from ch1
    if n_valid > 10:
        ch1_bottom = estimate_bottom_from_channel(ch1)
        ch1_valid = ch1_bottom >= 0
        both = valid & ch1_valid
        if both.sum() > 5:
            diffs = np.abs(bottom[both] - ch1_bottom[both]).astype(float)
            m['cross_freq_bottom_diff'] = float(np.mean(diffs))
        else:
            m['cross_freq_bottom_diff'] = np.nan
    else:
        m['cross_freq_bottom_diff'] = np.nan

    # --- Below-bottom decay ---
    # In a band 5-25 pixels below the bottom, intensity should decrease.
    # Compute the slope: negative = decaying (good), positive = increasing (bad).
    if n_valid > 10:
        slopes = []
        for j in np.where(valid)[0]:
            b = bottom[j]
            lo = b + 5
            hi = b + 25
            if hi < H:
                band_vals = ch0[lo:hi, j]
                if band_vals.std() > 1e-6:
                    x = np.arange(len(band_vals), dtype=float)
                    slope = np.polyfit(x, band_vals, 1)[0]
                    slopes.append(slope)
        if slopes:
            # Positive slope = intensity increasing below bottom = bad
            m['below_bottom_decay'] = float(np.mean(slopes))
        else:
            m['below_bottom_decay'] = np.nan
    else:
        m['below_bottom_decay'] = np.nan

    # --- Above-bottom coefficient of variation ---
    if n_valid > 10:
        water_pixels = []
        for j in np.where(valid)[0]:
            if bottom[j] > 10:
                water_pixels.append(ch0[:bottom[j], j])
        if water_pixels:
            water = np.concatenate(water_pixels)
            mean_w = water.mean()
            if mean_w > 1e-6:
                m['above_bottom_cv'] = float(water.std() / mean_w)
            else:
                m['above_bottom_cv'] = np.nan
        else:
            m['above_bottom_cv'] = np.nan
    else:
        m['above_bottom_cv'] = np.nan

    # --- Mask connectivity ---
    # Number of connected components in the binary mask (should be 1 for clean mask)
    if mask.any():
        labelled, n_components = ndimage.label(mask)
        m['mask_connectivity'] = int(n_components)
    else:
        m['mask_connectivity'] = 0

    # --- Bottom SNR ---
    # Difference in mean intensity between a band just below and just above the bottom.
    if n_valid > 10:
        band = 5
        snr_vals = []
        for j in np.where(valid)[0]:
            b = bottom[j]
            above_start = max(0, b - band)
            below_end = min(H, b + band)
            if b - above_start >= 2 and below_end - b >= 2:
                above_mean = ch0[above_start:b, j].mean()
                below_mean = ch0[b:below_end, j].mean()
                snr_vals.append(below_mean - above_mean)
        m['bottom_snr'] = float(np.mean(snr_vals)) if snr_vals else np.nan
    else:
        m['bottom_snr'] = np.nan

    return m


# Quick sanity check on one patch
d = np.load(npz_files[0])
test_metrics = compute_mask_metrics(d['patches'][0], d['masks'][0])
for k, v in test_metrics.items():
    if isinstance(v, float) and np.isnan(v):
        print(f'  {k:>26s}: NaN')
    else:
        print(f'  {k:>26s}: {v:.4f}' if isinstance(v, float) else f'  {k:>26s}: {v}')

## Compute metrics for all patches

Single pass over every NPZ file. Each patch gets a uid of the form `{npz_stem}::{patch_index}` so it can be cross-referenced with the notebook 05 metrics.

In [ ]:
records = []

for npz_path in tqdm(npz_files, desc='Computing mask metrics'):
    stem = npz_path.stem
    source_zarr = '_'.join(stem.split('_')[:-1])  # drop the hash suffix
    data = np.load(npz_path)
    patches, masks = data['patches'], data['masks']
    n = patches.shape[0]

    for i in range(n):
        uid = f'{stem}::{i}'
        m = compute_mask_metrics(patches[i], masks[i])
        m['uid'] = uid
        m['npz_file'] = stem
        m['patch_idx'] = i
        m['source_zarr'] = source_zarr
        m['height'] = patches.shape[2]
        records.append(m)

    del data, patches, masks

df = pd.DataFrame(records)
del records

id_cols = ['uid', 'npz_file', 'patch_idx', 'source_zarr', 'height']
metric_cols = [c for c in df.columns if c not in id_cols]
df = df[id_cols + metric_cols]

MASK_METRIC_COLS = [
    'bottom_continuity', 'bottom_curvature', 'mask_edge_intensity',
    'cross_freq_bottom_diff', 'below_bottom_decay', 'above_bottom_cv',
    'mask_connectivity', 'bottom_snr',
]

df.to_csv(OUTPUT_CSV, index=False)
print(f'Computed metrics for {len(df)} patches across {df["npz_file"].nunique()} files')
print(f'Saved to {OUTPUT_CSV}')
df.describe()

## Metric distributions

In [ ]:
ncols = 4
nrows = int(np.ceil(len(MASK_METRIC_COLS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
axes = axes.flat

for i, col in enumerate(MASK_METRIC_COLS):
    ax = axes[i]
    vals = df[col].dropna()
    ax.hist(vals, bins=50, color='steelblue', edgecolor='none', alpha=0.8)
    p5, p50, p95 = vals.quantile([0.05, 0.5, 0.95])
    ax.axvline(p5, color='red', ls='--', lw=0.8, label=f'p5={p5:.2f}')
    ax.axvline(p50, color='orange', ls='-', lw=0.8, label=f'p50={p50:.2f}')
    ax.axvline(p95, color='green', ls='--', lw=0.8, label=f'p95={p95:.2f}')
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=6)
    ax.set_ylabel('Count')

for j in range(i + 1, len(list(axes))):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## Correlation with notebook 05 metrics

If the general patch-quality CSV exists, merge and check how much new information the mask-specific metrics add. Low correlation with the existing composite score means these metrics capture failure modes that the general metrics miss.

In [ ]:
if NB05_CSV.exists():
    df_nb05 = pd.read_csv(NB05_CSV)
    # Merge on uid
    merged = df.merge(df_nb05[['uid', 'quality_score', 'bottom_coverage',
                                'bottom_smoothness', 'boundary_contrast',
                                'freq_coherence', 'bottom_gradient_max']],
                       on='uid', how='inner', suffixes=('', '_nb05'))
    print(f'Merged {len(merged)} patches with nb05 metrics')

    # Correlation matrix: new mask metrics vs selected nb05 metrics
    nb05_cols = ['quality_score', 'bottom_coverage', 'bottom_smoothness',
                 'boundary_contrast', 'freq_coherence', 'bottom_gradient_max']
    corr_cols = MASK_METRIC_COLS + nb05_cols
    corr = merged[corr_cols].corr()

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xticks(range(len(corr_cols)))
    ax.set_yticks(range(len(corr_cols)))
    ax.set_xticklabels(corr_cols, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(corr_cols, fontsize=8)
    for i in range(len(corr_cols)):
        for j in range(len(corr_cols)):
            v = corr.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=6,
                        color='white' if abs(v) > 0.6 else 'black')
    # Draw a divider line between new and nb05 metrics
    n_new = len(MASK_METRIC_COLS)
    ax.axhline(n_new - 0.5, color='black', lw=2)
    ax.axvline(n_new - 0.5, color='black', lw=2)
    fig.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('New mask metrics (top-left) vs nb05 metrics (bottom-right)')
    plt.tight_layout()
    plt.show()
else:
    print(f'No nb05 CSV at {NB05_CSV} — skipping correlation analysis.'
          f' Run notebook 05 first if you want this comparison.')

## Visualization helpers

In [ ]:
def load_patch_by_uid(uid, patch_dir=PATCH_DIR):
    """Load a single patch + mask from disk given its uid."""
    stem, idx_str = uid.split('::')
    idx = int(idx_str)
    data = np.load(patch_dir / f'{stem}.npz')
    return data['patches'][idx], data['masks'][idx]


def plot_patch_with_mask(patch, mask, uid, ax, zoom_bottom=True, margin=150,
                         subtitle=None):
    """Show ch0 echogram with bottom line overlay on a given axes."""
    ch0 = patch[0]
    bottom = get_bottom_line(mask)
    valid = bottom >= 0
    H, W = ch0.shape

    if zoom_bottom and valid.any():
        med = int(np.median(bottom[valid]))
        row_lo = max(0, med - margin)
        row_hi = min(H, med + margin)
        ch0_crop = ch0[row_lo:row_hi, :]
        bottom_shifted = np.where(valid, bottom - row_lo, -1)
    else:
        ch0_crop = ch0
        bottom_shifted = bottom
        row_lo = 0

    ax.imshow(ch0_crop, aspect='auto', cmap='viridis', vmin=0, vmax=1,
              interpolation='nearest')

    xs = np.arange(W)
    valid_shifted = bottom_shifted >= 0
    if valid_shifted.any():
        ax.plot(xs[valid_shifted], bottom_shifted[valid_shifted],
                'r-', lw=1, alpha=0.9)

    ax.set_title(uid, fontsize=7, fontfamily='monospace', pad=2)
    ax.set_xticks([])
    ax.set_yticks([])
    if subtitle:
        ax.set_xlabel(subtitle, fontsize=6, fontfamily='monospace',
                      color='0.4', labelpad=2)


def _get_subtitle(uid):
    """Look up source_zarr for a uid from the dataframe."""
    row = df.loc[df['uid'] == uid]
    if not row.empty:
        return row.iloc[0]['source_zarr']
    return None


def gallery(uids, title, ncols=4, zoom_bottom=True, margin=150):
    """Plot a grid of patches identified by their uids."""
    n = len(uids)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    if nrows == 1:
        axes = np.atleast_2d(axes)
    axes = axes.flat

    for i, uid in enumerate(uids):
        patch, mask = load_patch_by_uid(uid)
        plot_patch_with_mask(patch, mask, uid, axes[i],
                             zoom_bottom=zoom_bottom, margin=margin,
                             subtitle=_get_subtitle(uid))

    for j in range(n, len(list(axes))):
        axes[j].set_visible(False)

    fig.suptitle(title, fontsize=12, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

## Worst and best patches per metric

For each mask metric, the 8 patches with the worst value. Worst means low continuity, high curvature, low edge intensity, high cross-freq diff, positive decay slope, high CV, high connectivity count, and low SNR.

In [ ]:
# For each metric, whether the WORST patches have the highest or lowest values
WORST_IS = {
    'bottom_continuity':      'smallest',
    'bottom_curvature':       'largest',
    'mask_edge_intensity':    'smallest',
    'cross_freq_bottom_diff': 'largest',
    'below_bottom_decay':     'largest',   # positive slope = bad
    'above_bottom_cv':        'largest',
    'mask_connectivity':      'largest',   # more components = worse
    'bottom_snr':             'smallest',
}

N_SHOW = 8

for col in MASK_METRIC_COLS:
    sub = df.dropna(subset=[col])
    if WORST_IS[col] == 'largest':
        worst = sub.nlargest(N_SHOW, col)
    else:
        worst = sub.nsmallest(N_SHOW, col)
    vals = worst[col].values
    labels = [f'{uid}\n{col}={v:.3f}' for uid, v in zip(worst['uid'], vals)]
    gallery(worst['uid'].tolist(),
            f'WORST {N_SHOW} by {col} ({WORST_IS[col]})',
            ncols=4)

## Best patches per metric

Same layout but showing the 8 best patches per metric as a sanity check.

In [ ]:
for col in MASK_METRIC_COLS:
    sub = df.dropna(subset=[col])
    # Best is the opposite of worst
    if WORST_IS[col] == 'largest':
        best = sub.nsmallest(N_SHOW, col)
    else:
        best = sub.nlargest(N_SHOW, col)
    gallery(best['uid'].tolist(),
            f'BEST {N_SHOW} by {col}',
            ncols=4)

## Composite mask quality score

Each metric is rank-normalised to [0, 1] (higher = better mask), then combined with weights that reflect how reliably each metric identifies genuinely bad masks.

In [ ]:
MASK_METRIC_DIRECTION = {
    'bottom_continuity':      'higher_is_better',
    'bottom_curvature':       'lower_is_better',
    'mask_edge_intensity':    'higher_is_better',
    'cross_freq_bottom_diff': 'lower_is_better',
    'below_bottom_decay':     'lower_is_better',   # negative slope = good decay
    'above_bottom_cv':        'lower_is_better',
    'mask_connectivity':      'lower_is_better',   # 1 component is ideal
    'bottom_snr':             'higher_is_better',
}

MASK_METRIC_WEIGHTS = {
    'bottom_continuity':      1.5,
    'bottom_curvature':       1.5,
    'mask_edge_intensity':    2.0,  # most direct evidence of real bottom
    'cross_freq_bottom_diff': 1.5,
    'below_bottom_decay':     1.0,
    'above_bottom_cv':        0.5,
    'mask_connectivity':      1.0,
    'bottom_snr':             2.0,  # strong indicator
}


def rank_normalise(series, direction):
    """Rank-normalise a series to [0, 1] where 1 = best quality."""
    ranked = series.rank(pct=True, na_option='bottom')
    if direction == 'lower_is_better':
        ranked = 1.0 - ranked
    return ranked


score = np.zeros(len(df))
total_weight = 0

for col in MASK_METRIC_COLS:
    direction = MASK_METRIC_DIRECTION[col]
    weight = MASK_METRIC_WEIGHTS[col]
    rn = rank_normalise(df[col], direction)
    df[f'{col}_rn'] = rn
    score += weight * rn.fillna(0).values
    total_weight += weight

df['mask_quality_score'] = score / total_weight

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df['mask_quality_score'], bins=60, color='steelblue', edgecolor='none', alpha=0.8)
ax.set_xlabel('Composite mask quality score')
ax.set_ylabel('Count')
ax.set_title('Distribution of mask quality scores (higher = better)')
ax.axvline(df['mask_quality_score'].quantile(0.05), color='red', ls='--', label='p5 (worst 5%)')
ax.axvline(df['mask_quality_score'].quantile(0.95), color='green', ls='--', label='p95 (best 5%)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mask quality score range: [{df["mask_quality_score"].min():.3f}, {df["mask_quality_score"].max():.3f}]')
print(f'Worst 5% threshold: {df["mask_quality_score"].quantile(0.05):.3f}')
print(f'Best 5% threshold:  {df["mask_quality_score"].quantile(0.95):.3f}')

## Composite score: worst and best 20

In [ ]:
worst_20 = df.nsmallest(20, 'mask_quality_score')
gallery(worst_20['uid'].tolist(), 'WORST 20 by composite mask quality score', ncols=5)

best_20 = df.nlargest(20, 'mask_quality_score')
gallery(best_20['uid'].tolist(), 'BEST 20 by composite mask quality score', ncols=5)

## Per-file mask quality

Some entire .raw files may have systematically bad masks (e.g. noisy conditions, wrong threshold, shallow water). This groups patches by source file and identifies the worst files.

In [ ]:
file_summary = (
    df.groupby('source_zarr')
    .agg(
        n_patches=('uid', 'count'),
        mean_mask_quality=('mask_quality_score', 'mean'),
        mean_continuity=('bottom_continuity', 'mean'),
        mean_curvature=('bottom_curvature', 'mean'),
        mean_edge_intensity=('mask_edge_intensity', 'mean'),
        mean_cross_freq_diff=('cross_freq_bottom_diff', 'mean'),
        mean_snr=('bottom_snr', 'mean'),
        mean_connectivity=('mask_connectivity', 'mean'),
    )
    .sort_values('mean_mask_quality')
)

fig, ax = plt.subplots(figsize=(14, max(4, len(file_summary) * 0.35)))
colors = plt.cm.RdYlGn(file_summary['mean_mask_quality'] / file_summary['mean_mask_quality'].max())
ax.barh(range(len(file_summary)), file_summary['mean_mask_quality'], color=colors)
ax.set_yticks(range(len(file_summary)))
ax.set_yticklabels([f'{name} (n={int(row["n_patches"])})'
                     for name, row in file_summary.iterrows()],
                    fontsize=7, fontfamily='monospace')
ax.set_xlabel('Mean mask quality score')
ax.set_title('Per-file mask quality (sorted worst-first)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Worst 5 files:')
for name, row in file_summary.head(5).iterrows():
    print(f'  {name}: score={row["mean_mask_quality"]:.3f}, '
          f'n={int(row["n_patches"])}, '
          f'continuity={row["mean_continuity"]:.3f}, '
          f'snr={row["mean_snr"]:.3f}')

## Mask failure taxonomy

Categorize bad masks into failure types based on which metrics are extreme. A patch can belong to multiple categories. This helps understand whether bad masks share common root causes.

In [ ]:
# Classify patches into failure categories using percentile thresholds.
# A patch is flagged in a category if its metric falls in the worst 10%.

p10 = df[MASK_METRIC_COLS].quantile(0.10)
p90 = df[MASK_METRIC_COLS].quantile(0.90)

df['fail_gap_filled'] = (df['bottom_continuity'] < p10['bottom_continuity'])
df['fail_jagged'] = (df['bottom_curvature'] > p90['bottom_curvature'])
df['fail_no_boundary'] = (df['mask_edge_intensity'] < p10['mask_edge_intensity'])
df['fail_freq_disagree'] = (df['cross_freq_bottom_diff'] > p90['cross_freq_bottom_diff'])
df['fail_false_bottom'] = (df['below_bottom_decay'] > p90['below_bottom_decay'])
df['fail_noisy_water'] = (df['above_bottom_cv'] > p90['above_bottom_cv'])
df['fail_fragmented'] = (df['mask_connectivity'] > 1)
df['fail_low_snr'] = (df['bottom_snr'] < p10['bottom_snr'])

fail_cols = [c for c in df.columns if c.startswith('fail_')]
fail_labels = [c.replace('fail_', '') for c in fail_cols]

# Count how many patches fall into each category
counts = df[fail_cols].sum().values

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(fail_labels, counts, color='coral', edgecolor='none', alpha=0.8)
ax.set_ylabel('Number of patches')
ax.set_title('Mask failure categories (p10/p90 thresholds)')
ax.set_xticklabels(fail_labels, rotation=30, ha='right')
for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            str(int(count)), ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

# How many patches have multiple failure modes?
df['n_failures'] = df[fail_cols].sum(axis=1)
print('Patches by number of failure categories:')
print(df['n_failures'].value_counts().sort_index().to_string())
print(f'\nPatches with 3+ failure modes: {(df["n_failures"] >= 3).sum()}')

## Gallery: patches with 3+ failure modes

These are the most suspicious masks — flagged by multiple independent metrics.

In [ ]:
multi_fail = df[df['n_failures'] >= 3].nsmallest(20, 'mask_quality_score')
if len(multi_fail) > 0:
    gallery(multi_fail['uid'].tolist(),
            f'Patches with 3+ failure modes ({len(multi_fail)} shown, '
            f'{(df["n_failures"] >= 3).sum()} total)',
            ncols=5)
else:
    print('No patches have 3+ failure modes.')

## Export flagged masks

Two auto-flagging strategies: patches in the worst 5% by composite score, and patches with 3+ failure modes. You can also paste specific uids into the manual list. The output JSON can be loaded during training to exclude these patches.

In [ ]:
# ---- Paste bad uids here ----
manual_flagged = [
    # 'D20240131-T092044_721e57f964fb::3',
]

# Auto-flag: worst 5% by composite score
p5_threshold = df['mask_quality_score'].quantile(0.05)
auto_worst = set(df.loc[df['mask_quality_score'] < p5_threshold, 'uid'].tolist())

# Auto-flag: 3+ failure modes
auto_multi_fail = set(df.loc[df['n_failures'] >= 3, 'uid'].tolist())

all_flagged = sorted(set(manual_flagged) | auto_worst | auto_multi_fail)

print(f'Manual flags:      {len(manual_flagged)}')
print(f'Auto (worst 5%):   {len(auto_worst)}')
print(f'Auto (3+ fails):   {len(auto_multi_fail)}')
print(f'Union (total):     {len(all_flagged)}')

with open(FLAGGED_JSON, 'w') as f:
    json.dump({'flagged_uids': all_flagged}, f, indent=2)
print(f'\nSaved to {FLAGGED_JSON}')

## Paginated browser

Sorted by mask quality score (worst first). Change `PAGE` and re-run to browse through all patches.

In [ ]:
PAGE = 0
PER_PAGE = 20

sorted_df = df.sort_values('mask_quality_score')
start = PAGE * PER_PAGE
end = start + PER_PAGE
page_uids = sorted_df.iloc[start:end]['uid'].tolist()

gallery(page_uids,
        f'Page {PAGE} (patches {start}-{end-1}, worst first)',
        ncols=5)

## Save full results

Re-save the CSV with the composite score and failure flags included.

In [ ]:
df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved {len(df)} rows to {OUTPUT_CSV}')
print(f'Columns: {list(df.columns)}')